#Load bronze -> silver 

In [0]:
cards_df = spark.table("jrvs_databricks_fundamentals.bronze.cards_data")
transactions_df = spark.table("jrvs_databricks_fundamentals.bronze.transactions_data")
user_df = spark.table("jrvs_databricks_fundamentals.bronze.users_data")
mcc_df = spark.table("jrvs_databricks_fundamentals.bronze.mcc_codes")
fraud_df = spark.table("jrvs_databricks_fundamentals.bronze.fraud_labels")

## Transformation

###Helper Functions 

In [0]:
from pyspark.sql.functions import col, trim, regexp_replace


def trim_strings(df):
    # trim leading/trailing spaces on every text column
    for field in df.schema.fields:
        if field.dataType.typeName() == "string":
            df = df.withColumn(field.name, trim(col(field.name)))
    return df

###Cards Data

In [0]:
cards_df.printSchema()
display(cards_df.limit(10))



root
 |-- id: short (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: short (nullable = true)
 |-- has_chip: boolean (nullable = true)
 |-- num_cards_issued: short (nullable = true)
 |-- credit_limit: decimal(19,4) (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: short (nullable = true)
 |-- card_on_dark_web: string (nullable = true)



id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,true,2,24295.0000,09/2002,2008,No
2731,825,Visa,Debit,4956965974959986,12/2020,393,true,2,21968.0000,04/2014,2014,No
3701,825,Visa,Debit,4582313478255491,02/2024,719,true,2,46414.0000,07/2003,2004,No
42,825,Visa,Credit,4879494103069057,08/2024,693,false,1,12400.0000,01/2003,2012,No
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,true,1,28.0000,09/2008,2009,No
4537,1746,Visa,Credit,4404898874682993,09/2003,736,true,1,27500.0000,09/2003,2012,No
1278,1746,Visa,Debit,4001482973848631,07/2022,972,true,2,28508.0000,02/2011,2011,No
3687,1746,Mastercard,Debit,5627220683410948,06/2022,48,true,2,9022.0000,07/2003,2015,No
3465,1746,Mastercard,Debit (Prepaid),5711382187309326,11/2020,722,true,2,54.0000,06/2010,2015,No
3754,1746,Mastercard,Debit (Prepaid),5766121508358701,02/2023,908,true,1,99.0000,07/2006,2012,No


In [0]:
from pyspark.sql.functions import to_date, date_format, col,trim



silver_cards_df = (
    cards_df.dropDuplicates(["id"])
    .withColumnRenamed("id", "card_id")
    # keep MM/yyyy but validate it via date round-trip
    .withColumn("expires", date_format(to_date(col("expires"), "MM/yyyy"), "MM/yyyy"))
    .withColumn("acct_open_date", date_format(to_date(col("acct_open_date"), "MM/yyyy"), "MM/yyyy"))
    .withColumn("credit_limit", col("credit_limit").cast("decimal(19,2)"))
    # create a boolean column for whether the card is on the dark web
    .withColumn("is_on_dark_web", col("card_on_dark_web") == "Yes")
    .drop("card_on_dark_web")
)
#clean white spaces
trim_strings(silver_cards_df)

silver_cards_df.printSchema()
display(silver_cards_df.limit(10))

root
 |-- card_id: short (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: short (nullable = true)
 |-- has_chip: boolean (nullable = true)
 |-- num_cards_issued: short (nullable = true)
 |-- credit_limit: decimal(19,2) (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: short (nullable = true)
 |-- is_on_dark_web: boolean (nullable = true)



card_id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,is_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,true,2,24295.00,09/2002,2008,false
2731,825,Visa,Debit,4956965974959986,12/2020,393,true,2,21968.00,04/2014,2014,false
3701,825,Visa,Debit,4582313478255491,02/2024,719,true,2,46414.00,07/2003,2004,false
42,825,Visa,Credit,4879494103069057,08/2024,693,false,1,12400.00,01/2003,2012,false
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,true,1,28.00,09/2008,2009,false
4537,1746,Visa,Credit,4404898874682993,09/2003,736,true,1,27500.00,09/2003,2012,false
1278,1746,Visa,Debit,4001482973848631,07/2022,972,true,2,28508.00,02/2011,2011,false
3687,1746,Mastercard,Debit,5627220683410948,06/2022,48,true,2,9022.00,07/2003,2015,false
3465,1746,Mastercard,Debit (Prepaid),5711382187309326,11/2020,722,true,2,54.00,06/2010,2015,false
3754,1746,Mastercard,Debit (Prepaid),5766121508358701,02/2023,908,true,1,99.00,07/2006,2012,false


###Transaction Data

In [0]:
transactions_df.printSchema()
display(transactions_df.limit(10))

root
 |-- id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_id: short (nullable = true)
 |-- amount: decimal(19,4) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: short (nullable = true)
 |-- errors: string (nullable = true)



id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
18207094,2016-08-10T06:24:00.000Z,38,2526,30.4200,Online Transaction,39021,ONLINE,null,null,4784,null
18207096,2016-08-10T06:24:00.000Z,1456,5935,140.0000,Chip Transaction,27092,Lucedale,MS,39452.0,4829,null
18207097,2016-08-10T06:24:00.000Z,1586,1122,18.2200,Chip Transaction,94123,Woodmere,NY,11598.0,5310,null
18207098,2016-08-10T06:24:00.000Z,1746,1278,17.5400,Chip Transaction,22204,Little Neck,NY,11363.0,5541,null
18207099,2016-08-10T06:25:00.000Z,52,4976,2.1200,Chip Transaction,79927,Pawling,NY,12564.0,5912,null
18207100,2016-08-10T06:25:00.000Z,190,3851,2.2700,Chip Transaction,75781,Waukegan,IL,60085.0,5411,null
18207101,2016-08-10T06:25:00.000Z,1490,5950,11.3600,Swipe Transaction,61195,Commerce City,CO,80022.0,5541,null
18207102,2016-08-10T06:25:00.000Z,1989,2984,13.2600,Swipe Transaction,95690,Bethesda,MD,20817.0,4131,null
18207103,2016-08-10T06:26:00.000Z,162,5938,57.0000,Chip Transaction,59935,West Bloomfield,MI,48322.0,5499,null
18207104,2016-08-10T06:26:00.000Z,428,1351,66.5800,Chip Transaction,36159,Manchester,NH,3104.0,5411,null


In [0]:
from pyspark.sql.functions import col, lit, when, trim, lower, upper,to_timestamp, to_date, hour, date_format

silver_transactions_df = (
    transactions_df
    .withColumn("amount", col("amount").cast("decimal(19,2)"))
    .withColumn("date", to_timestamp("date"))
    .withColumn(
        "use_chip",
        when(trim(col("use_chip")) == "Online Transaction", lit("online"))
        .when(trim(col("use_chip")) == "Chip Transaction", lit("chip"))
        .when(trim(col("use_chip")) == "Swipe Transaction", lit("swipe"))
        .otherwise(lower(trim(col("use_chip"))))
    )
    # online transactions: flag them, keep the nulls
    .withColumn(
        "is_online",
        (col("use_chip") == "online") | (upper(trim(col("merchant_city"))) == "ONLINE")
    )
    # non-online rows with missing geo -> map to 'unknown' instead of null
    .withColumn(
        "merchant_state",
        when(~col("is_online") & col("merchant_state").isNull(), lit("unknown"))
        .otherwise(col("merchant_state"))
    )
)

silver_transactions_df = trim_strings(silver_transactions_df)

silver_transactions_df.printSchema()
display(silver_transactions_df.limit(10))

root
 |-- id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_id: short (nullable = true)
 |-- amount: decimal(19,2) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: short (nullable = true)
 |-- errors: string (nullable = true)
 |-- is_online: boolean (nullable = true)



id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,is_online
7475327,2010-01-01T00:01:00.000Z,1556,2972,-77.00,swipe,59935,Beulah,ND,58523.0,5499,null,false
7475328,2010-01-01T00:02:00.000Z,561,4575,14.57,swipe,67570,Bettendorf,IA,52722.0,5311,null,false
7475329,2010-01-01T00:02:00.000Z,1129,102,80.00,swipe,27092,Vista,CA,92084.0,4829,null,false
7475331,2010-01-01T00:05:00.000Z,430,2860,200.00,swipe,27092,Crown Point,IN,46307.0,4829,null,false
7475332,2010-01-01T00:06:00.000Z,848,3915,46.41,swipe,13051,Harwood,MD,20776.0,5813,null,false
7475333,2010-01-01T00:07:00.000Z,1807,165,4.81,swipe,20519,Bronx,NY,10464.0,5942,null,false
7475334,2010-01-01T00:09:00.000Z,1556,2972,77.00,swipe,59935,Beulah,ND,58523.0,5499,null,false
7475335,2010-01-01T00:14:00.000Z,1684,2140,26.46,online,39021,ONLINE,null,null,4784,null,true
7475336,2010-01-01T00:21:00.000Z,335,5131,261.58,online,50292,ONLINE,null,null,7801,null,true
7475337,2010-01-01T00:21:00.000Z,351,1112,10.74,swipe,3864,Flushing,NY,11355.0,5813,null,false


###User Data


In [0]:
user_df.printSchema()
display(user_df.limit(10))

root
 |-- id: short (nullable = true)
 |-- current_age: short (nullable = true)
 |-- retirement_age: short (nullable = true)
 |-- birth_year: short (nullable = true)
 |-- birth_month: short (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: decimal(19,4) (nullable = true)
 |-- yearly_income: decimal(19,4) (nullable = true)
 |-- total_debt: decimal(19,4) (nullable = true)
 |-- credit_score: short (nullable = true)
 |-- num_credit_cards: short (nullable = true)



id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.150001525878906,-117.76000213623047,29278.0000,59696.0000,127613.0000,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.7599983215332,-73.73999786376953,37891.0000,77254.0000,191349.0000,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02000045776367,-117.88999938964844,22681.0000,33483.0000,196.0000,698,5
708,63,63,1957,1,Female,3 Madison Street,40.709999084472656,-73.98999786376953,163145.0000,249925.0000,202328.0000,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.7599983215332,-122.44000244140625,53797.0000,109687.0000,183855.0000,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.54999923706055,-90.5999984741211,20599.0000,41997.0000,0.0000,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.220001220703125,-85.73999786376953,25258.0000,51500.0000,102286.0000,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.5099983215332,-122.63999938964844,26790.0000,54623.0000,114711.0000,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.31999969482422,-75.31999969482422,26273.0000,42509.0000,2895.0000,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.969999313354492,-92.12000274658203,18730.0000,38190.0000,81262.0000,810,1


In [0]:
from pyspark.sql.functions import col, when, lit

silver_user_df = (
    user_df
    .dropDuplicates(["id"])
    .withColumnRenamed("id", "user_id")
    # money: decimal(19,4) -> decimal(19,2)
    .withColumn("per_capita_income", col("per_capita_income").cast("decimal(19,2)"))
    .withColumn("yearly_income", col("yearly_income").cast("decimal(19,2)"))
    .withColumn("total_debt", col("total_debt").cast("decimal(19,2)"))

    # look for  0/0 placeholder from the originals before we null 
    .withColumn("_bad_coords", (col("latitude") == 0) & (col("longitude") == 0))

    # latitude: null if out of range or 0/0 placeholder
    .withColumn(
        "latitude",
        when(col("_bad_coords") | (col("latitude") < -90) | (col("latitude") > 90), lit(None))
        .otherwise(col("latitude"))
    )
    # longitude: null if out of range or 0/0 placeholder
    .withColumn(
        "longitude",
        when(col("_bad_coords") | (col("longitude") < -180) | (col("longitude") > 180), lit(None))
        .otherwise(col("longitude"))
    )
    .drop("_bad_coords")

    # birth_month: valid 1-12, else null
    .withColumn(
        "birth_month",
        when((col("birth_month") < 1) | (col("birth_month") > 12), lit(None))
        .otherwise(col("birth_month"))
    )
    # birth_year: sane range, else null
    .withColumn(
        "birth_year",
        when((col("birth_year") < 1900) | (col("birth_year") > 2026), lit(None))
        .otherwise(col("birth_year"))
    )


)

# apply your trim helper last
silver_user_df = trim_strings(silver_user_df)

silver_user_df.printSchema()
display(silver_user_df.limit(10))

root
 |-- user_id: short (nullable = true)
 |-- current_age: short (nullable = true)
 |-- retirement_age: short (nullable = true)
 |-- birth_year: short (nullable = true)
 |-- birth_month: short (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: decimal(19,2) (nullable = true)
 |-- yearly_income: decimal(19,2) (nullable = true)
 |-- total_debt: decimal(19,2) (nullable = true)
 |-- credit_score: short (nullable = true)
 |-- num_credit_cards: short (nullable = true)



user_id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.150001525878906,-117.76000213623047,29278.00,59696.00,127613.00,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.7599983215332,-73.73999786376953,37891.00,77254.00,191349.00,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02000045776367,-117.88999938964844,22681.00,33483.00,196.00,698,5
708,63,63,1957,1,Female,3 Madison Street,40.709999084472656,-73.98999786376953,163145.00,249925.00,202328.00,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.7599983215332,-122.44000244140625,53797.00,109687.00,183855.00,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.54999923706055,-90.5999984741211,20599.00,41997.00,0.00,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.220001220703125,-85.73999786376953,25258.00,51500.00,102286.00,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.5099983215332,-122.63999938964844,26790.00,54623.00,114711.00,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.31999969482422,-75.31999969482422,26273.00,42509.00,2895.00,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.969999313354492,-92.12000274658203,18730.00,38190.00,81262.00,810,1


### MCC Codes

In [0]:
from pyspark.sql.functions import col

silver_mcc_df = (
    mcc_df
    .withColumn("mcc_code", col("mcc_code").cast("int"))
    .dropDuplicates(["mcc_code"])
)

silver_mcc_df = trim_strings(silver_mcc_df)
silver_mcc_df.printSchema()
display(silver_mcc_df.limit(10))

root
 |-- mcc_code: integer (nullable = true)
 |-- description: string (nullable = true)



mcc_code,description
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products
3007,Coated and Laminated Products
3008,Steel Drums and Barrels
3009,Fabricated Structural Metal Products
3058,"Tools, Parts, Supplies Manufacturing"
3066,Miscellaneous Metals


### Fraud Labels

In [0]:
from pyspark.sql.functions import col, trim, lower, when, lit

silver_fraud_df = (
    fraud_df
    # match transactions.id type for the join
    .withColumn("transaction_id", col("transaction_id").cast("int"))

    # Yes/No -> boolean is_fraud
    .withColumn(
        "is_fraud",
        when(lower(trim(col("label"))) == "yes", lit(True))
        .when(lower(trim(col("label"))) == "no", lit(False))
        .otherwise(lit(None).cast("boolean"))
    )
    .drop("label")
    .dropDuplicates(["transaction_id"])
)

silver_fraud_df.printSchema()
display(silver_fraud_df.limit(10))

root
 |-- transaction_id: integer (nullable = true)
 |-- is_fraud: boolean (nullable = true)



transaction_id,is_fraud
10649266,false
23410063,false
9316588,false
12478022,false
9558530,false
12532830,false
19526714,false
9906964,false
13224888,false
13749094,false
